# Praktikum 3 - Klasifikasi Multinomial Naive Bayes pada Teks SMS (`spam.csv`)

**Nama:** Mohammad Al Hafis Hidayatulloh  
**NIM:** 254107023005  
**Mata Kuliah:** RTI235004 - Pembelajaran Mesin  
**Dosen Pengampu:** Usman Nurhasan, S.Kom., M.T.

---

## Langkah 1 - Load Data (`spam.csv`)

Membaca dataset `spam.csv` menggunakan `encoding='latin-1'` karena terdapat karakter khusus non-UTF-8 di dalam pesan SMS.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv('spam.csv', encoding='latin-1')  # spesifikasi encoding diperlukan karena data tidak menggunakan UTF-8
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


## Langkah 2 - Preprocessing

### Langkah 2a - Drop Kolom dan Rename Kolom

Membuang 3 kolom terakhir yang kosong (`Unnamed: 2`, `Unnamed: 3`, `Unnamed: 4`) menggunakan `iloc[:, 2:]` serta mengubah nama kolom `v1` menjadi `Labels` dan `v2` menjadi `SMS`.

In [2]:
# Drop 3 kolom terakhir dengan fungsi iloc
df = df.drop(df.iloc[:, 2:], axis=1)

# Ubah nama kolom v1 dan v2
df = df.rename(columns={'v1': 'Labels', 'v2': 'SMS'})

# Cek data
df.head()

,Labels,SMS
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Langkah 2b - Inspeksi Data

Mengecek jumlah pesan per kelas (`ham` vs `spam`), kelengkapan data (`df.info()`), dan statistik deskriptif (`df.describe()`).

In [3]:
# Cek Jumlah Data Per Kelas
print(df['Labels'].value_counts())
print('\n')

# Cek Kelengkapan Data
print(df.info())
print('\n')

# Cek Statistik Deskriptif
print(df.describe())

Labels
ham     4825
spam     747
Name: count, dtype: int64


<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Labels  5572 non-null   str  
 1   SMS     5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB
None


       Labels                     SMS
count    5572                    5572
unique      2                    5169
top       ham  Sorry, I'll call later
freq     4825                      30


### Langkah 2c - Encoding Label

Mengubah nilai teks pada kolom `Labels` menjadi nilai biner (`'spam': 1`, `'ham': 0`).

In [4]:
# Data untuk label
new_labels = {
    'spam': 1,
    'ham': 0
}

# Encode label
df['Labels'] = df['Labels'].map(new_labels)

# Cek data
df.head()

,Labels,SMS
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


### Langkah 2d - Pisahkan Fitur dengan Label

Memisahkan array teks `SMS` sebagai `X` dan array biner `Labels` sebagai `y`.

In [5]:
X = df['SMS'].values
y = df['Labels'].values

## Langkah 3 - Ekstraksi Fitur (*Bag of Words* dengan `CountVectorizer`)

Membagi data terlebih dahulu dengan rasio `80:20` (`random_state = 50`) untuk mencegah *data leakage*, kemudian melakukan ekstraksi fitur frekuensi kemunculan kata (*Bag of Words*) menggunakan `CountVectorizer`.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Split data training dan testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

# Inisiasi CountVectorizer
bow = CountVectorizer()

# Fitting dan transform X_train dengan CountVectorizer
X_train = bow.fit_transform(X_train)

# Transform X_test
X_test = bow.transform(X_test)

print(len(bow.get_feature_names_out()))
print(f'Dimensi data: {X_train.shape}')

7727
Dimensi data: (4457, 7727)


## Langkah 4 - Training dan Evaluasi Model (`MultinomialNB`)

Melatih model `MultinomialNB` pada matriks *Bag of Words* `X_train` dan mengevaluasi akurasinya pada data latih maupun data uji.

In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Inisiasi MultinomialNB
mnb = MultinomialNB()

# Fit model
mnb.fit(X_train, y_train)

# Prediksi dengan data training
y_pred_train = mnb.predict(X_train)

# Evaluasi akurasi data training
acc_train = accuracy_score(y_train, y_pred_train)

# Prediksi dengan data testing
y_pred_test = mnb.predict(X_test)

# Evaluasi akurasi data testing
acc_test = accuracy_score(y_test, y_pred_test)

# Print hasil evaluasi
print(f'Hasil akurasi data train: {acc_train}')
print(f'Hasil akurasi data test: {acc_test}')
print('\nConfusion Matrix (Data Test):\n', confusion_matrix(y_test, y_pred_test))

Hasil akurasi data train: 0.9946152120260264
Hasil akurasi data test: 0.9775784753363229

Confusion Matrix (Data Test):
 [[951   3]
 [ 22 139]]


### Analisis Hasil Praktikum 3

Dari hasil ekstraksi fitur menggunakan `CountVectorizer`, seluruh `4.457` SMS pada data latih menghasilkan perbendaharaan kata (*vocabulary*) sebanyak `7.727` kata unik sehingga terbentuk matriks jarang (*sparse matrix*) berukuran `(4457, 7727)`. Penting bahwa `bow.fit_transform()` hanya dipanggil pada `X_train`, sedangkan `X_test` (`1.115` SMS) hanya dikenakan `bow.transform()` supaya kamus kata dari data uji tidak bocor (*data leakage*) ke proses pelatihan. Ketika dilatih menggunakan `MultinomialNB`—yang memang dirancang khusus untuk data cacahan frekuensi diskrit (*word counts*)—model mencapai akurasi yang sangat tinggi yaitu **`0.9946` (`99.46%`) pada data latih** dan **`0.9776` (`97.76%`) pada data uji**.